In [ ]:
from random import randrange, choice
def req(method, id, val):
    o = [method]
    if method not in ('ls','du'):
        if isinstance(id, tuple):
            o += id
        else:
            o.append(id)
        if method=='set' or method.startswith('math'):
            if isinstance(val, tuple):
                o += val
            else:
                o.append(val)
    return ' '.join(map(str, o))
def rsp(s, method, id, val, ring):
    if isinstance(id, tuple):
        if _ := s.get(id[0]) is None:
            id,t = id
        else:
            id = id[0]
            t = _
    else:
        t = s.get(id)
    if isinstance(val, tuple):
        if _ := s.get(val[0]) is None:
            val = val[1]
        else:
            val = _
    elif isinstance(val, str):
        val = s.get(val)
    if method == 'ls':
        return 'ok' + ''.join(f' {i} {j}'for i,j in s.items()if j is not None)
    elif method == 'du':
        return f'ok {len(s)}'
    elif method == 'echo':
        return f'ok {id}'
    elif method == 'get':
        val = t
    elif method == 'set':
        s[id] = val
    elif method == 'del':
        s[id] = val = None
    elif method.startswith('math'):
        if t is None or val is None:
            s[id] = t = None
        elif method == 'mathadd':
            s[id] = t = (t + val) % ring
        elif method == 'mathsub':
            s[id] = t = (t - val) % ring
        elif method == 'mathmul':
            s[id] = t = (t * val) % ring
        elif val == 0:
            s[id] = t = float('nan')
        elif method == 'mathdiv':
            s[id] = t = t // val
        elif method == 'mathmod':
            s[id] = t = t % val
        else:
            return f'nocmd {method}'
    else:
        return f'nocmd {method}'
    return f'ok {"nul"if t is None else t} {"nul"if val is None else val}'
def dat(n=1, m=1, var='ab', ring=1000):
    A = []
    B = []
    s = [{}for m in range(m)]
    for n in range(n):
        a = []
        b = []
        for m in s:
            method = choice(('ls','du','echo','get',*('set',)*20,'del','mathadd','mathsub','mathmul','mathdiv','mathmod'))
            id = randrange(2)
            if id == 0:
                id = choice(var)
            else:
                id = choice(var), randrange(ring)
            val = randrange(3)
            if val == 0:
                val = randrange(ring)
            elif val == 1:
                val = choice(var)
            else:
                val = choice(var), randrange(ring)
            a.append(req(method, id, val))
            b.append(rsp(m, method, id, val, ring))
        A.append(a)
        B.append(b)
    return A, B
def strs(a):
    if isinstance(a, str):
        yield a
    else:
        for a in a:
            yield from strs(a)
def char(a):
    for a in strs(a):
        yield from a
def maxlen(a):
    return max(map(len, strs(a)))

In [ ]:
import torch
from torch.nn import *
class Lang:
    def __init__(s, a):
        s.a = '<bos>','<eos>',*set(a)
        s.A = {a:i for i,a in enumerate(s.a)}
    def __len__(s):
        return len(s.a)
    def __call__(s, a, d='cuda'):
        return torch.tensor([[[0if i<0else s.A[a[i]]if i<len(a)else 1for a in a]for a in a]for i in range(-1,maxlen(a)+1)], device=d)
    def __getitem__(s, a):
        o = []
        for a in a.permute(1,2,0).tolist():
            T = []
            for a in a:
                t = []
                for a in a:
                    if a == 1:
                        break
                    t.append(s.a[a])
                T.append(''.join(t))
            o.append(T)
        return o
class Seq2Seq(Module):
    def __init__(s, e, k, K, h=1):
        super().__init__()
        s._emb = Embedding(e, k)
        s._enc = GRU(k, K)
        s._nn = Sequential(Linear(K, h*K), ReLU(1), Linear(h*K, K))
        s._dec = GRU(k, K)
        s._deb = Linear(K, k)
        s.ph = Linear(K, K)
    def _los(s, y, t):
        return functional.cross_entropy(y.permute(0,y.ndim-1,*range(1,y.ndim-1)), t)
    def emb(s, x, k=.25):
        x = s._emb(x)
        if k:
            x[torch.rand(x.shape[:-1])<k] = 0
        return x
    def enc(s, x):
        return s._enc(x)[1]
    def dec(s, x, t):
        return s._dec(t, x)[0]
    def nn(s, x):
        return s._nn(x)
    def deb(s, x):
        return s._deb(x) @ s._emb.weight.T
    def forward(s, x, t, **_):
        return s.deb(s.dec(s.nn(s.enc(s.emb(x[1:-1], **_))), s.emb(t[:-2])))
    def los(s, x, t):
        x = x[:-1]
        t = t[:-1]
        embx = s.emb(x)
        embt = s.emb(t)
        encx = s.enc(embx[1:])
        y = s.deb(s.dec(s.nn(encx), embt[:-1]))
        l = s._los(y, t[1:]) + s._los(s.deb(s.dec(s.ph(encx), embx[:-1])), x[1:]) + s._los(s.deb(s.dec(s.ph(s.enc(embt[1:])), embt[:-1])), t[1:])
        return y, l
class Seq2Rnn2Seq(Seq2Seq):
    def __init__(s, e, k, K, **_):
        super().__init__(e, k, K, **_)
        s._rnn = GRU(K, K)
    def enc(s, x):
        return super().enc(x.flatten(1,2))[0].unflatten(0,x.shape[1:3])
    def nn(s, x):
        return super().nn(s._rnn(x)[0])
    def dec(s, x, t):
        return super().dec(x.flatten(end_dim=1)[None], t.flatten(1,2)).unflatten(1, x.shape[:2])

In [ ]:
from tabulate import tabulate
from IPython.display import HTML
a = Lang(char(dat(999)))
m = Seq2Rnn2Seq(len(a), 4, 64).cuda()
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, .003, 200000)
i = 1
for i in range(i, O.total_steps):
    x,t = dat(32, 32)
    y,l = m.los(a(x), a(t))
    l.backward()
    o.step()
    o.zero_grad()
    O.step()
    if not i % 500:
        print(end=f'{l:.3f} ')
        if not i % 10000:
            display(HTML(tabulate([[i for i in zip(*i)for i in i]for i in zip(x,t,a[y[:,:8,:4].argmax(3)])], tablefmt='html')))